# LeetCode 671 — Second Minimum Node In a Binary Tree bằng Python

**Link đề:** https://leetcode.com/problems/second-minimum-node-in-a-binary-tree/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search, Binary Tree

```python
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def findSecondMinimumValue(self, root: Optional[TreeNode]) -> int:
        ...
```

## Mục tiêu học
1. Dựng và in cây nhị phân từ list kiểu LeetCode; duyệt cây bằng DFS đệ quy, stack, BFS.
2. Giải bài **tổng quát** (cây bất kỳ): gom giá trị vào `set`, tìm số nhỏ thứ hai.
3. Khai thác **tính chất đặc biệt**: `node.val == min(left.val, right.val)` ⇒ **gốc là giá trị nhỏ nhất**.
4. **Cắt tỉa** (pruning) DFS: gặp nút có `val > root.val` thì **không cần đi sâu hơn**.
5. Xử lý trường hợp "không tồn tại" → trả `-1`; tránh dùng giá trị "vô cực" sai cách.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho một cây nhị phân **đặc biệt**, không rỗng, các giá trị không âm:
- mỗi nút có **đúng 0 hoặc 2** con;
- nếu có 2 con thì `node.val = min(node.left.val, node.right.val)`.

Trả về **giá trị nhỏ thứ hai** trong **tập** giá trị của mọi nút. Nếu không có → `-1`.

**Ví dụ 1:** `root = [2,2,5,null,null,5,7]` → `5`
```text
      2
     / \
    2   5
       / \
      5   7
```
Tập giá trị `{2, 5, 7}` → nhỏ nhất 2, nhỏ thứ hai **5**.

**Ví dụ 2:** `root = [2,2,2]` → `-1` (tập chỉ có `{2}`).

**Constraints:** số nút trong `[1, 25]`; `1 <= Node.val <= 2^31 - 1`; tính chất min đúng với mọi nút trong.

Lưu ý: "nhỏ thứ hai trong **tập**" → các giá trị **trùng** chỉ tính một lần (`[2,2,5]` cho 5 chứ không phải 2).

## 2. Nền tảng: `TreeNode`, dựng cây, duyệt cây

### 2.1. Lớp `TreeNode` và hàm dựng/in cây

In [1]:
from typing import Optional
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: list) -> Optional[TreeNode]:
    # Dựng cây từ list theo tầng kiểu LeetCode (None = trống).
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root


def print_tree(node, prefix="", label="root"):
    if node is None:
        return
    print(prefix + f"{label}: {node.val}")
    print_tree(node.left, prefix + "    ", "L")
    print_tree(node.right, prefix + "    ", "R")


root1 = build_tree([2, 2, 5, None, None, 5, 7])
print_tree(root1)

root: 2
    L: 2
    R: 5
        L: 5
        R: 7


### 2.2. DFS đệ quy — thăm mọi nút
Đệ quy: xử lý nút hiện tại, rồi gọi lại cho con trái và con phải. **Điều kiện dừng**: `node is None`.

In [2]:
def all_values(node: Optional[TreeNode]) -> list[int]:
    if node is None:
        return []
    return [node.val] + all_values(node.left) + all_values(node.right)

print("pre-order:", all_values(root1))

pre-order: [2, 2, 5, 5, 7]


### 2.3. `set` và `sorted` — tập không trùng, sắp xếp tăng
`sorted(set(values))` bỏ trùng rồi sắp tăng. Phần tử thứ hai (chỉ số 1) là số nhỏ thứ hai — **nếu** tập có ≥ 2 phần tử.

In [3]:
vals = [2, 2, 5, 5, 7]
uniq = sorted(set(vals))
print("set:", set(vals), "| sorted:", uniq)
print("nhỏ thứ hai:", uniq[1] if len(uniq) >= 2 else -1)
print("với [2,2,2]:", sorted(set([2, 2, 2])))

set: {2, 5, 7} | sorted: [2, 5, 7]
nhỏ thứ hai: 5
với [2,2,2]: [2]


### 2.4. `float('inf')` làm giá trị "chưa tìm thấy"
Khi tìm **min**, khởi tạo `ans = float('inf')` — mọi số đều nhỏ hơn. Cuối cùng nếu `ans` vẫn là `inf` → không tìm thấy → trả `-1`.
⚠ Không dùng `2**31 - 1` làm "vô cực" vì giá trị nút **có thể đúng bằng** `2^31 - 1`!

In [4]:
ans = float('inf')
for v in [7, 5, 9]:
    ans = min(ans, v)
print(ans)
print("inf có lớn hơn 2**31-1 không?", float('inf') > 2**31 - 1)

5
inf có lớn hơn 2**31-1 không? True


### 2.5. Tính chất của cây đặc biệt: **gốc là nhỏ nhất**
Vì mỗi nút trong bằng `min` của hai con, giá trị **không giảm** khi đi từ gốc xuống lá. Do đó:
- `root.val` = giá trị nhỏ nhất của cả cây.
- Với mọi nút `x`, mọi giá trị trong cây con của `x` đều `≥ x.val`.

In [5]:
def check_property(node) -> bool:
    if node is None or node.left is None:
        return True
    ok = node.val == min(node.left.val, node.right.val)
    return ok and check_property(node.left) and check_property(node.right)

print("cây 1 thỏa tính chất?", check_property(root1))
print("gốc =", root1.val, "| min toàn cây =", min(all_values(root1)))

cây 1 thỏa tính chất? True
gốc = 2 | min toàn cây = 2


## 3. Cách trực giác — gom tất cả giá trị, sắp xếp

1. Gom mọi giá trị vào `set`.
2. Sắp xếp; nếu có ≥ 2 phần tử, trả phần tử thứ hai; ngược lại `-1`.

Cách này **không dùng** tính chất đặc biệt nên đúng cho mọi cây. Time `O(n log n)`, Space `O(n)`.

In [6]:
def second_min_sorting(root: Optional[TreeNode]) -> int:
    uniq = sorted(set(all_values(root)))
    return uniq[1] if len(uniq) >= 2 else -1

print(second_min_sorting(build_tree([2, 2, 5, None, None, 5, 7])))   # 5
print(second_min_sorting(build_tree([2, 2, 2])))                     # -1

5
-1


### 3.1. Không cần sắp xếp: tìm min hai lần
Cải tiến nhỏ: `m1 = min(tập)`, rồi `m2 = min(v for v in tập if v > m1)` → `O(n)`.
Hoặc theo dõi `first, second` trong **một lượt** như bài "Third Maximum Number".

In [7]:
def second_min_two_pass(root):
    vals = all_values(root)
    m1 = min(vals)
    bigger = [v for v in vals if v > m1]
    return min(bigger) if bigger else -1

def second_min_one_pass(root):
    first = second = float('inf')
    for v in all_values(root):
        if v < first:
            first, second = v, first
        elif first < v < second:
            second = v
    return second if second != float('inf') else -1

for vals in ([2, 2, 5, None, None, 5, 7], [2, 2, 2], [1, 1, 3, 1, 1, 3, 4]):
    t = build_tree(vals)
    print(vals, "->", second_min_two_pass(t), second_min_one_pass(t))

[2, 2, 5, None, None, 5, 7] -> 5 5
[2, 2, 2] -> -1 -1
[1, 1, 3, 1, 1, 3, 4] -> 3 3


## 4. Nút thắt và insight

**Nút thắt:** các cách trên phải thăm **mọi** nút và chưa dùng tính chất đặc biệt.

**Insight 1:** `root.val` chắc chắn là nhỏ nhất → đáp án = **giá trị nhỏ nhất trong các giá trị `> root.val`**.

**Insight 2 (cắt tỉa):** nếu gặp nút `x` có `x.val > root.val`, thì mọi giá trị trong cây con của `x` đều `≥ x.val` → `x.val` là **ứng viên tốt nhất** của cả cây con đó → **không cần đi sâu** vào `x` nữa.

Chỉ những nút có `val == root.val` mới cần đi tiếp xuống (vì bên dưới có thể có giá trị lớn hơn nhưng nhỏ hơn các ứng viên khác).

In [8]:
root2 = build_tree([2, 2, 5, None, None, 5, 7])
print("root.val =", root2.val)
print("nút phải có val =", root2.right.val, "> root.val -> cây con của nó toàn >=", root2.right.val)
print("giá trị trong cây con phải:", all_values(root2.right))

root.val = 2
nút phải có val = 5 > root.val -> cây con của nó toàn >= 5
giá trị trong cây con phải: [5, 5, 7]


## 5. Lời giải chính — DFS có cắt tỉa (giảng từng đoạn)

**Đoạn 1:** `min1 = root.val`, `ans = float('inf')`.

**Đoạn 2:** hàm `dfs(node)`:
- `node is None` → return.
- nếu `min1 < node.val < ans` → `ans = node.val`, **return** (cắt tỉa, không đi sâu).
- nếu `node.val == min1` → `dfs(node.left)`, `dfs(node.right)`.
- (nếu `node.val >= ans` → cũng không cần đi sâu, vì cây con toàn giá trị ≥ ans.)

**Đoạn 3:** sau DFS, trả `ans` nếu đã tìm được, ngược lại `-1`.

Dùng `nonlocal ans` để hàm con **gán lại** biến của hàm ngoài.

In [9]:
def outer():
    count = 0
    def inc():
        nonlocal count      # không có dòng này -> UnboundLocalError
        count += 1
    inc(); inc()
    return count

print("nonlocal demo:", outer())

nonlocal demo: 2


In [10]:
def second_min_pruned(root: Optional[TreeNode]) -> int:
    min1 = root.val                     # đoạn 1
    ans = float('inf')

    def dfs(node):                      # đoạn 2
        nonlocal ans
        if node is None:
            return
        if min1 < node.val < ans:
            ans = node.val              # ứng viên mới; cây con không có gì tốt hơn
            return
        if node.val == min1:
            dfs(node.left)
            dfs(node.right)

    dfs(root)
    return ans if ans < float('inf') else -1   # đoạn 3

print(second_min_pruned(build_tree([2, 2, 5, None, None, 5, 7])))  # 5
print(second_min_pruned(build_tree([2, 2, 2])))                    # -1
print(second_min_pruned(build_tree([5])))                          # -1

5
-1
-1


## 6. TRACE từng bước

Cây `[1, 1, 3, 1, 1, 3, 4, 3, 1, 1, 1, 3, 8, 4, 8, 3, 3, 1, 6, 2, 1]` — cây lớn có nhiều nhánh bằng `min1 = 1`. Đáp án là **2** nằm khá sâu.

Ta in mỗi lần thăm: nút, độ sâu, quyết định (đi sâu / ứng viên / bỏ qua).

In [11]:
def trace_second_min(root: Optional[TreeNode]) -> int:
    min1 = root.val
    ans = float('inf')
    visits = 0

    def dfs(node, depth):
        nonlocal ans, visits
        if node is None:
            return
        visits += 1
        pad = "  " * depth
        if min1 < node.val < ans:
            ans = node.val
            print(f"{pad}{node.val}: > min1 và < ans cũ -> ans = {ans}, CẮT TỈA")
            return
        if node.val == min1:
            print(f"{pad}{node.val}: == min1 -> đi xuống hai con")
            dfs(node.left, depth + 1)
            dfs(node.right, depth + 1)
        else:
            print(f"{pad}{node.val}: >= ans = {ans} -> bỏ qua cả cây con")

    dfs(root, 0)
    res = ans if ans < float('inf') else -1
    print(f"Số nút đã thăm: {visits} | kết quả: {res}")
    return res

big = build_tree([1, 1, 3, 1, 1, 3, 4, 3, 1, 1, 1, 3, 8, 4, 8, 3, 3, 1, 6, 2, 1])
print("tổng số nút:", len(all_values(big)))
trace_second_min(big)

tổng số nút: 21
1: == min1 -> đi xuống hai con
  1: == min1 -> đi xuống hai con
    1: == min1 -> đi xuống hai con
      3: > min1 và < ans cũ -> ans = 3, CẮT TỈA
      1: == min1 -> đi xuống hai con
        1: == min1 -> đi xuống hai con
        6: >= ans = 3 -> bỏ qua cả cây con
    1: == min1 -> đi xuống hai con
      1: == min1 -> đi xuống hai con
        2: > min1 và < ans cũ -> ans = 2, CẮT TỈA
        1: == min1 -> đi xuống hai con
      1: == min1 -> đi xuống hai con
  3: >= ans = 2 -> bỏ qua cả cây con
Số nút đã thăm: 13 | kết quả: 2


2

Quan sát: chỉ 13/21 nút được thăm; toàn bộ cây con dưới nút `3` (con phải của gốc) và dưới nút `6` **không bị thăm** nhờ cắt tỉa. Với ví dụ 1 nhỏ:

| Bước | Nút | So sánh | Quyết định | `ans` |
|---:|---:|---|---|---:|
| 1 | 2 (gốc) | `== min1` | đi xuống | ∞ |
| 2 | 2 (trái) | `== min1` | đi xuống (hai con None) | ∞ |
| 3 | 5 (phải) | `2 < 5 < ∞` | ứng viên, cắt tỉa | 5 |

Hai nút `5, 7` bên dưới không bao giờ được thăm.

In [12]:
trace_second_min(build_tree([2, 2, 5, None, None, 5, 7]))

2: == min1 -> đi xuống hai con
  2: == min1 -> đi xuống hai con
  5: > min1 và < ans cũ -> ans = 5, CẮT TỈA
Số nút đã thăm: 3 | kết quả: 5


5

## 7. Code nộp LeetCode

### 7.1. Cách chính — DFS cắt tỉa

In [13]:
class Solution:
    def findSecondMinimumValue(self, root: Optional[TreeNode]) -> int:
        min1 = root.val
        ans = float('inf')

        def dfs(node):
            nonlocal ans
            if node is None:
                return
            if min1 < node.val < ans:
                ans = node.val
            elif node.val == min1:
                dfs(node.left)
                dfs(node.right)

        dfs(root)
        return ans if ans < float('inf') else -1

print(Solution().findSecondMinimumValue(build_tree([2, 2, 5, None, None, 5, 7])))
print(Solution().findSecondMinimumValue(build_tree([2, 2, 2])))

5
-1


### 7.2. Cách đệ quy "thuần" — trả về giá trị thay vì `nonlocal`

Định nghĩa `f(node)` = giá trị nhỏ nhất `> min1` trong cây con của `node` (hoặc `-1`).
- `node.val > min1` → trả `node.val` (cắt tỉa).
- lá có `val == min1` → `-1`.
- ngược lại: `a = f(left)`, `b = f(right)`; nếu cả hai khác `-1` trả `min(a, b)`, nếu không trả cái khác `-1` (hoặc `-1`).

In [14]:
class SolutionRecursive:
    def findSecondMinimumValue(self, root: Optional[TreeNode]) -> int:
        min1 = root.val

        def f(node) -> int:
            if node is None:
                return -1
            if node.val > min1:
                return node.val
            a, b = f(node.left), f(node.right)
            if a != -1 and b != -1:
                return min(a, b)
            return a if a != -1 else b

        return f(root)

print(SolutionRecursive().findSecondMinimumValue(build_tree([2, 2, 5, None, None, 5, 7])))
print(SolutionRecursive().findSecondMinimumValue(build_tree([2, 2, 2])))

5
-1


### 7.3. Cách BFS (không đệ quy), có cắt tỉa

In [15]:
class SolutionBFS:
    def findSecondMinimumValue(self, root: Optional[TreeNode]) -> int:
        min1, ans = root.val, float('inf')
        q = deque([root])
        while q:
            node = q.popleft()
            if min1 < node.val < ans:
                ans = node.val
            elif node.val == min1 and node.left:
                q.append(node.left)
                q.append(node.right)
        return ans if ans < float('inf') else -1

print(SolutionBFS().findSecondMinimumValue(build_tree([2, 2, 5, None, None, 5, 7])))

5


### 7.4. Cách tổng quát (không dùng tính chất) — set + sort
Đúng cho mọi cây, phù hợp nếu đề "lỡ" bỏ tính chất min.

In [16]:
class SolutionSet:
    def findSecondMinimumValue(self, root: Optional[TreeNode]) -> int:
        vals = set()
        stack = [root]
        while stack:
            node = stack.pop()
            if node:
                vals.add(node.val)
                stack.append(node.left)
                stack.append(node.right)
        uniq = sorted(vals)
        return uniq[1] if len(uniq) > 1 else -1

print(SolutionSet().findSecondMinimumValue(build_tree([2, 2, 5, None, None, 5, 7])))

5


## 8. Vì sao đúng?

**Bổ đề:** với mọi nút `x`, mọi giá trị trong cây con của `x` đều `≥ x.val`. (Quy nạp từ lá lên: `x.val = min(left.val, right.val) ≤` mọi giá trị trong hai cây con.)

Hệ quả: `min1 = root.val` là nhỏ nhất toàn cây; đáp án = `min{v : v > min1}`.

**Cắt tỉa đúng:**
- Khi gặp `x` với `x.val > min1`: mọi giá trị trong cây con `≥ x.val`, nên ứng viên tốt nhất của cây con là `x.val` → không mất gì khi dừng.
- Khi gặp `x` với `x.val ≥ ans` (và `> min1`): cây con không thể chứa giá trị `< ans` → bỏ qua an toàn.
- Chỉ đi sâu khi `x.val == min1` — đây là những nút mà bên dưới có thể có giá trị lớn hơn `min1` mà ta chưa thấy.

Mọi giá trị `> min1` trong cây hoặc được xét trực tiếp, hoặc nằm trong cây con của một nút đã xét có giá trị `≤` nó → `ans` cuối cùng là min đúng.

## 9. Độ phức tạp và so sánh

Gọi `n` = số nút (≤ 25), `h` = chiều cao.

| Cách | Time | Space | Dùng tính chất? |
|---|---:|---:|:---:|
| Gom + sort | `O(n log n)` | `O(n)` | Không |
| Gom + 2 lần min / 1 lượt | `O(n)` | `O(n)` | Không |
| DFS cắt tỉa (`nonlocal`) | `O(n)` tệ nhất, thường ít hơn | `O(h)` | **Có** |
| Đệ quy trả giá trị | `O(n)` | `O(h)` | **Có** |
| BFS cắt tỉa | `O(n)` | `O(n)` (hàng đợi) | **Có** |

Tệ nhất (mọi nút bằng `min1`) cắt tỉa vẫn phải thăm hết → `O(n)`.

## 10. Bộ test

Cần sinh **cây đặc biệt hợp lệ** để test ngẫu nhiên: dựng từ lá lên — mỗi nút trong lấy `min` của hai con.

In [17]:
import random

def random_special_tree(rng, depth):
    # Sinh cây đặc biệt: lá có giá trị ngẫu nhiên, nút trong = min hai con.
    if depth == 0 or rng.random() < 0.3:
        return TreeNode(rng.randint(1, 6))
    left = random_special_tree(rng, depth - 1)
    right = random_special_tree(rng, depth - 1)
    return TreeNode(min(left.val, right.val), left, right)

cases = [
    ([2, 2, 5, None, None, 5, 7], 5),
    ([2, 2, 2], -1),
    ([5], -1),
    ([1, 1, 3, 1, 1, 3, 4, 3, 1, 1, 1, 3, 8, 4, 8, 3, 3, 1, 6, 2, 1], 2),
    ([2, 2, 2147483647], 2147483647),     # giá trị lớn nhất có thể
    ([1, 1, 1, 1, 2], 2),                  # đáp án nằm sâu dưới nhánh trái
]
solvers = [Solution().findSecondMinimumValue, SolutionRecursive().findSecondMinimumValue,
           SolutionBFS().findSecondMinimumValue, SolutionSet().findSecondMinimumValue,
           second_min_pruned, second_min_one_pass]
for vals, expected in cases:
    t = build_tree(vals)
    assert check_property(t)
    for f in solvers:
        assert f(t) == expected, (f, vals)
    print(f"{str(vals)[:50]:<52} -> {expected} PASS")

rng = random.Random(671)
for _ in range(500):
    t = random_special_tree(rng, 4)
    exp = second_min_sorting(t)
    assert all(f(t) == exp for f in solvers)
print("500 cây đặc biệt ngẫu nhiên: mọi cách khớp PASS")

[2, 2, 5, None, None, 5, 7]                          -> 5 PASS
[2, 2, 2]                                            -> -1 PASS
[5]                                                  -> -1 PASS
[1, 1, 3, 1, 1, 3, 4, 3, 1, 1, 1, 3, 8, 4, 8, 3, 3   -> 2 PASS
[2, 2, 2147483647]                                   -> 2147483647 PASS
[1, 1, 1, 1, 2]                                      -> 2 PASS
500 cây đặc biệt ngẫu nhiên: mọi cách khớp PASS


## 11. Lỗi người mới hay gặp

### Lỗi 1: Trả `min(root.left.val, root.right.val)` hoặc `max(...)`
Chỉ nhìn tầng 2 → sai khi đáp án nằm sâu hơn (cả hai con đều bằng gốc).

In [18]:
def wrong_look_children(root):
    if root.left is None:
        return -1
    vals = {root.left.val, root.right.val} - {root.val}
    return min(vals) if vals else -1

t = build_tree([1, 1, 1, 1, 2])
print("sai:", wrong_look_children(t), "| đúng:", Solution().findSecondMinimumValue(t))

sai: -1 | đúng: 2


### Lỗi 2: Không bỏ trùng
Sắp xếp **list** (không phải set) rồi lấy `[1]` → `[2, 2, 5]` cho `2` thay vì `5`.

### Lỗi 3: Dùng `2**31 - 1` làm "vô cực"
Khi đáp án thật là `2147483647`, ta không phân biệt được "chưa tìm thấy" với "tìm thấy giá trị max" → trả `-1` sai.

In [19]:
def wrong_no_dedup(root):
    vals = sorted(all_values(root))
    return vals[1] if len(vals) > 1 else -1

print("sai:", wrong_no_dedup(build_tree([2, 2, 5, None, None, 5, 7])), "| đúng: 5")

def wrong_int_max(root):
    INF = 2**31 - 1
    min1, ans = root.val, INF
    def dfs(node):
        nonlocal ans
        if node is None:
            return
        if min1 < node.val < ans:
            ans = node.val
        elif node.val == min1:
            dfs(node.left); dfs(node.right)
    dfs(root)
    return ans if ans < INF else -1

t = build_tree([2, 2, 2147483647])
print("sai:", wrong_int_max(t), "| đúng:", Solution().findSecondMinimumValue(t))

sai: 2 | đúng: 5
sai: -1 | đúng: 2147483647


### Lỗi 4: Quên `nonlocal`
Gán `ans = node.val` bên trong hàm con mà không khai báo `nonlocal ans` → Python coi `ans` là biến **cục bộ mới** → `UnboundLocalError` ở phép so sánh trước đó.

### Lỗi 5: Cắt tỉa sai — dừng cả khi `node.val == min1`
Nếu `return` ngay khi gặp nút bằng `min1` thì bỏ lỡ đáp án nằm sâu dưới nó.

In [20]:
def wrong_missing_nonlocal(root):
    min1, ans = root.val, float('inf')
    def dfs(node):
        if node is None:
            return
        if min1 < node.val < ans:     # ans ở đây bị coi là biến cục bộ chưa gán
            ans = node.val
        elif node.val == min1:
            dfs(node.left); dfs(node.right)
    dfs(root)
    return ans

try:
    wrong_missing_nonlocal(build_tree([2, 2, 5]))
except UnboundLocalError as e:
    print("UnboundLocalError:", e)

UnboundLocalError: cannot access local variable 'ans' where it is not associated with a value


## 12. Mẹo nhận dạng pattern

- Đề cho **tính chất đặc biệt** của cây (BST, heap, "nút = min hai con"...) → hỏi ngay: tính chất này cho phép **cắt tỉa** nhánh nào?
- Cây "nút = min con" chính là một **min-heap** dạng cây: gốc nhỏ nhất, đi xuống không giảm.
- "Giá trị nhỏ/lớn thứ k **phân biệt**" → nhớ **bỏ trùng**; với k = 2 theo dõi `first`, `second` một lượt.
- Giá trị "chưa có" → `float('inf')`/`None`, tránh "số ma thuật" có thể trùng với dữ liệu thật.

## 13. Biến thể

### 13.1. Giá trị nhỏ thứ k phân biệt trên cây bất kỳ

In [21]:
import heapq

def kth_smallest_distinct(root, k):
    uniq = set(all_values(root))
    if len(uniq) < k:
        return -1
    return heapq.nsmallest(k, uniq)[-1]

t = build_tree([2, 2, 5, None, None, 5, 7])
print([kth_smallest_distinct(t, k) for k in (1, 2, 3, 4)])   # [2, 5, 7, -1]

[2, 5, 7, -1]


### 13.2. Third Maximum Number (LeetCode 414) — cùng kỹ thuật theo dõi top-k phân biệt

In [22]:
def third_max(nums):
    first = second = third = None
    for v in nums:
        if v in (first, second, third):
            continue
        if first is None or v > first:
            first, second, third = v, first, second
        elif second is None or v > second:
            second, third = v, second
        elif third is None or v > third:
            third = v
    return third if third is not None else first

print(third_max([3, 2, 1]), third_max([1, 2]), third_max([2, 2, 3, 1]))   # 1 2 1

1 2 1


## 14. Bài tập (tự làm trước, đáp án ở cuối)

**Bài 1 — Tính tay:** với cây `[3, 3, 4, 3, 3, None, None, 3, 6]` (kiểm tra xem có thỏa tính chất), đáp án là gì?

**Bài 2 — Đếm nút được thăm:** với ví dụ 1 `[2,2,5,null,null,5,7]`, cách DFS cắt tỉa thăm bao nhiêu nút **khác None**?

**Bài 3 — TODO:** viết hàm trả về **giá trị lớn nhất** của cây đặc biệt — có cần thăm hết không?

In [23]:
def exercise_max_value(root: Optional[TreeNode]) -> int:
    # TODO: duyệt cây (DFS hoặc BFS), trả max
    pass

# assert exercise_max_value(build_tree([2, 2, 5, None, None, 5, 7])) == 7
# assert exercise_max_value(build_tree([2, 2, 2])) == 2

**Bài 4 — TODO:** viết `exercise_second_min_stack` bằng **stack** (DFS không đệ quy) có cắt tỉa.

In [24]:
def exercise_second_min_stack(root: Optional[TreeNode]) -> int:
    # TODO: stack = [root]; pop; nếu min1 < val < ans -> cập nhật; nếu val == min1 và có con -> push 2 con
    pass

# assert exercise_second_min_stack(build_tree([2, 2, 5, None, None, 5, 7])) == 5
# assert exercise_second_min_stack(build_tree([2, 2, 2])) == -1

**Bài 5 — TODO:** viết hàm kiểm tra một cây có phải "cây đặc biệt" hợp lệ không (0 hoặc 2 con, nút = min hai con).

In [25]:
def exercise_is_special(root: Optional[TreeNode]) -> bool:
    # TODO: lá -> True; chỉ 1 con -> False; 2 con -> kiểm tra min và đệ quy
    pass

# assert exercise_is_special(build_tree([2, 2, 5, None, None, 5, 7])) is True
# assert exercise_is_special(build_tree([2, 3, 5])) is False
# assert exercise_is_special(build_tree([2, 2])) is False

**Bài 6 — Mini challenge:** trả về **số nút** có giá trị bằng giá trị nhỏ thứ hai (hoặc `0` nếu không tồn tại).

In [26]:
def exercise_count_second_min(root: Optional[TreeNode]) -> int:
    # TODO: tìm second = findSecondMinimumValue; đếm số nút có val == second
    pass

# assert exercise_count_second_min(build_tree([2, 2, 5, None, None, 5, 7])) == 2
# assert exercise_count_second_min(build_tree([2, 2, 2])) == 0

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Vẽ cây theo tầng | `min1 = 3` | ứng viên: 4 (nhánh phải), 6 (sâu) |
| 2 | Theo bảng trace mục 6 | Gốc, con trái, con phải | Hai nút `5, 7` bị cắt |
| 3 | Max có thể ở bất kỳ lá nào | Tính chất chỉ nói về min | Phải thăm hết: `O(n)` |
| 4 | Copy `SolutionBFS`, đổi deque → list | `stack.pop()` | `elif node.val == min1 and node.left:` |
| 5 | Ba trường hợp số con | `(node.left is None) != (node.right is None)` → False | `node.val == min(...)` và đệ quy hai bên |
| 6 | Hai pha: tìm rồi đếm | `if second == -1: return 0` | `sum(1 for v in all_values(root) if v == second)` |

## 16. Đáp án tham khảo

**Bài 1:** cây có gốc 3, hai con 3 và 4; con trái 3 có hai con 3, 3; con trái-trái 3 có con 3 và 6. Thỏa tính chất. Giá trị `{3, 4, 6}` → đáp án **4**.

**Bài 2:** thăm **3** nút khác None: gốc 2, con trái 2, con phải 5 (bị cắt tỉa ở đây).

In [27]:
t1 = build_tree([3, 3, 4, 3, 3, None, None, 3, 6])
print_tree(t1)
print("thỏa tính chất:", check_property(t1), "| Bài 1:", Solution().findSecondMinimumValue(t1))

root: 3
    L: 3
        L: 3
            L: 3
            R: 6
        R: 3
    R: 4
thỏa tính chất: True | Bài 1: 4


In [28]:
def answer_max_value(root):
    best, stack = root.val, [root]
    while stack:
        node = stack.pop()
        if node:
            best = max(best, node.val)
            stack.append(node.left); stack.append(node.right)
    return best


def answer_second_min_stack(root):
    min1, ans = root.val, float('inf')
    stack = [root]
    while stack:
        node = stack.pop()
        if min1 < node.val < ans:
            ans = node.val
        elif node.val == min1 and node.left:
            stack.append(node.left); stack.append(node.right)
    return ans if ans < float('inf') else -1


def answer_is_special(node):
    if node is None:
        return True
    if node.left is None and node.right is None:
        return True
    if node.left is None or node.right is None:
        return False
    return (node.val == min(node.left.val, node.right.val)
            and answer_is_special(node.left) and answer_is_special(node.right))


def answer_count_second_min(root):
    second = Solution().findSecondMinimumValue(root)
    if second == -1:
        return 0
    return sum(1 for v in all_values(root) if v == second)


assert answer_max_value(build_tree([2, 2, 5, None, None, 5, 7])) == 7
assert answer_max_value(build_tree([2, 2, 2])) == 2
assert answer_second_min_stack(build_tree([2, 2, 5, None, None, 5, 7])) == 5
assert answer_second_min_stack(build_tree([2, 2, 2])) == -1
assert answer_is_special(build_tree([2, 2, 5, None, None, 5, 7])) is True
assert answer_is_special(build_tree([2, 3, 5])) is False
assert answer_is_special(build_tree([2, 2])) is False
assert answer_count_second_min(build_tree([2, 2, 5, None, None, 5, 7])) == 2
assert answer_count_second_min(build_tree([2, 2, 2])) == 0
for _ in range(300):
    t = random_special_tree(rng, 4)
    assert answer_is_special(t)
    assert answer_second_min_stack(t) == second_min_sorting(t)
print("Đáp án tham khảo: tất cả assert PASS")

Đáp án tham khảo: tất cả assert PASS


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def findSecondMinimumValue(self, root: Optional[TreeNode]) -> int:
        min1, ans = root.val, float('inf')
        def dfs(node):
            nonlocal ans
            if node is None:
                return
            if min1 < node.val < ans:
                ans = node.val            # ứng viên, cắt tỉa
            elif node.val == min1:
                dfs(node.left)            # chỉ đi sâu khi bằng min1
                dfs(node.right)
        dfs(root)
        return ans if ans < float('inf') else -1
```

**Câu thần chú:** *"Gốc là min; tìm min của những gì lớn hơn gốc; lớn hơn thì dừng."*

**Checklist:**
- [ ] Bỏ trùng: nhỏ thứ hai **phân biệt**.
- [ ] Trả `-1` khi mọi nút bằng nhau / cây một nút.
- [ ] Dùng `float('inf')`, không dùng `2**31 - 1`.
- [ ] Có `nonlocal` khi gán biến ngoài trong hàm con.
- [ ] Không dừng ở nút bằng `min1` — đáp án có thể nằm sâu dưới nó.